In [64]:
from importlib import reload

import os
import sys
import json
from pathlib import Path
from .header import logger

In [10]:
#####################################################################
# Archive and unarchive a directory of r, py, yaml and ipynb files
########################################################################


def strip_outputs_from_ipynb(file_path):
    """
    Remove the output from a jupyter notebook an preserve only the code.
    """
    with open(file_path, 'r') as file:
        notebook = json.load(file)
    for cell in notebook.get('cells', []):
        if cell['cell_type'] == 'code':
            cell['outputs'] = []
    return json.dumps(notebook, indent=4)

def concatenate_files(directory, combined_file_path, file_types = ['.yaml', '.py', '.r']):
    """
    Send the directory you want archived.
    Returns a structured text string containing the archive
    This text string can be written to a archive file or copied to the buffer
    """
    print("in concatenate_files")
    all_contents = ""
    for root, dirs, files in os.walk(directory):
        for filename in files:
            # Skip files starting with '.' or '#'
            if filename.startswith('.') or filename.startswith('#'):
                continue

            if filename.endswith('.ipynb'):
                print(f"archiving {filename}")
                file_path = os.path.join(root, filename)
                try:
                    content = strip_outputs_from_ipynb(file_path)
                except FileNotFoundError:
                    print(f"File not found: {file_path}")
                    continue
            elif os.path.splitext(filename)[1] in file_types:
                print(f'filename is {filename}')
                file_path = os.path.join(root, filename)
                try:
                    with open(file_path, 'r') as file:
                        content = file.read()
                except FileNotFoundError:
                    print(f"File not found: {file_path}")
                    continue
            else:
                continue
            all_contents += f"---\nFilename: {filename}\n---\n{content}\n\n"

    # Write all_contents to the combined_file_path
    with open(combined_file_path, 'w') as file:
        file.write(all_contents)



def unpack_files(combined_file_path, output_directory):
    """
    Given a text file produced by `concatenate_files` unpack to a given `output_directory`
    

    Args:
        combined_file_path (_type_): _description_
        output_directory (_type_): _description_
    """
    with open(combined_file_path, 'r') as file:
        combined_content = file.read()

    # Split the content based on the delimiter used when combining the files
    sections = combined_content.split('---\nFilename: ')[1:]

    for section in sections:
        filename, content = section.split('\n---\n', 1)
        filename = filename.strip()
        output_path = os.path.join(output_directory, filename)
        
        with open(output_path, 'w') as file:
            file.write(content)
# Concatenate the files
def run_concat(current_directory, combined_files = 'combined_files.txt', file_types = ['.yaml', '.py', '.r']):
    """
    Wrapper for  the `concatenate_files` function
    """
    
    concatenate_files(current_directory, combined_files, file_types = file_types)
        
def run_upack(combined_file_path, output_directory):
    """
    Interface to the `unpack_files`
    The value for current directory can be gotting with
    `current_directory = os.getcwd()`
    """
    
    # Unpack the files
    unpack_files(combined_file_path, output_directory)

    print("Files have been unpacked into the directory:", output_directory)


def archive_subdirectories(parent_directory, directories=None, 
                           combined_archive_dir=None,
                           combined_archive_name='all_combined_archives.txt',
                           file_types = ['.yaml', '.py', '.r']):
    # If no specific directories are provided, use all subdirectories
    if directories is None:
        directories = os.listdir(parent_directory)

    # Create the combined_archive_dir if it doesn't exist
    if combined_archive_dir is not None:
        os.makedirs(combined_archive_dir, exist_ok=True)

    # Iterate over all items in the provided list
    for item in directories:
        item_path = os.path.join(parent_directory, item)
        print(f"Checking item: {item_path}")  # Debug print statement

        # Check if the item is a directory
        if os.path.isdir(item_path):
            print(f"{item_path} is a directory")  # Debug print statement
            archive_file_name = f"{item}_{combined_archive_name}.txt"
            # Define the name of the archive file for each subdirectory
            if combined_archive_dir is not None:
                archive_file = os.path.join(combined_archive_dir, archive_file_name)
            else:
                archive_file = os.path.join(parent_directory, archive_file_name)
            
            # Run the concatenation and archive process for each subdirectory
            run_concat(item_path, combined_files=archive_file, file_types = file_types)
            print(f"Archived {item_path} into {archive_file}")
        else:
            print(f"{item_path} is not a directory")  # Debug print statement

    # After all subdirectories have been archived, combine all archives into one
    combine_all_archives(parent_directory, combined_archive_dir, combined_archive_name, directories)      

def combine_all_archives(parent_directory, combined_archive_dir=None, 
                         combined_archive_name='all_combined_archives.txt', directories=None):
    print("function: combine_all_archives")
    all_archives_content = ""

    # If no specific directories are provided, use all subdirectories
    if directories is None:
        print(f"using All directories of {parent_directory}")
        directories = os.listdir(parent_directory)
    else:
        print(f"only using the directories: {directories}") 

    # Iterate over all items in parent directory
    for item in directories:
        # Define the name of the archive file for each subdirectory
        archive_file_name = f"{item}_{combined_archive_name}.txt"
        archive_file = os.path.join(combined_archive_dir or parent_directory, archive_file_name)
        
        # Check if the archive file exists and read its content
        if os.path.isfile(archive_file):
            with open(archive_file, 'r') as file:
                print(f"Extract Archived {archive_file} \n")
                content = file.read()
                print(f"---\nDirectory: {item}\n")
                all_archives_content += f"---\nDirectory: {item}\n---\n{content}\n\n"
                print("End of Directory\n")
        else:
            print(f"No archive file found for directory {item}")

    # Write all archives content to a single file in the parent directory
    if not combined_archive_name.endswith('.txt'):
        combined_archive_name += '.txt'
    
    combined_archive_path = os.path.join(combined_archive_dir or parent_directory, combined_archive_name)
    with open(combined_archive_path, 'w') as file:
        file.write(all_archives_content)

    print(f"All archives combined into {combined_archive_path}")
    
def unpack_all_archives(parent_directory, combined_archive_name='all_combined_archives.txt', overwrite = True):
    
    print(f"Starting to upack using {parent_directory} and {combined_archive_name}")
    combined_archive_path = os.path.join(parent_directory, combined_archive_name)

    # Read the combined archive file
    with open(combined_archive_path, 'r') as file:
        combined_content = file.read()

    # Split the content based on the directory delimiter
    directory_sections = combined_content.split('---\nDirectory: ')[1:]
    #print(f"the directory sections are {directory_sections} \n")

    for section in directory_sections:
        # Extract directory name and content
        directory_name, content = section.split('\n---\n', 1)
        print(f"restoring {directory_name}")
        directory_path = os.path.join(parent_directory, directory_name.strip())

        # Create the subdirectory if it doesn't exist
        os.makedirs(directory_path, exist_ok=True)
        
        
        archive_file_path = os.path.join(directory_path, 'combined_files.txt')

        # Check if 'combined_files.txt' already exists
        if os.path.exists(archive_file_path) and not overwrite:
            archive_file_path = os.path.join(directory_path, 'combined_files_copy.txt')

        # Write content to 'combined_files.txt' in the subdirectory
        with open(archive_file_path, 'w') as file:
            file.write(content)

        # Now unpack the files in the subdirectory
        unpack_files(archive_file_path, directory_path)

        print(f"Unpacked archive in {directory_path}")

# Example usage
#parent_directory = '/path/to/Projects'
#unpack_all_archives(parent_directory)



In [66]:
parentDirectory = Path(os.getcwd(),'Projects')
parentDirectory

WindowsPath('d:/research/healtheintent/Projects')

In [61]:
#reload(a)
unpack_all_archives(parentDirectory, combined_archive_name='Projects.txt')

Starting to upack using d:\research\healtheintent\Projects and Projects.txt
restoring Projects
Unpacked archive in d:\research\healtheintent\Projects\Projects
restoring .ipynb_checkpoints
Unpacked archive in d:\research\healtheintent\Projects\.ipynb_checkpoints
restoring ACE_COVID_CREATININE
Unpacked archive in d:\research\healtheintent\Projects\ACE_COVID_CREATININE
restoring Alzheimer-smoking
Unpacked archive in d:\research\healtheintent\Projects\Alzheimer-smoking
restoring Cohort_Extraction
Unpacked archive in d:\research\healtheintent\Projects\Cohort_Extraction
restoring Colon-Cancer
Unpacked archive in d:\research\healtheintent\Projects\Colon-Cancer
restoring Emergency
Unpacked archive in d:\research\healtheintent\Projects\Emergency
restoring Experimental
Unpacked archive in d:\research\healtheintent\Projects\Experimental
restoring Kidney Function
Unpacked archive in d:\research\healtheintent\Projects\Kidney Function
restoring OABLaPorte
Unpacked archive in d:\research\healtheinten

## Read in a archived file.

### To archive

`find foresight/*.py -type f -exec echo "FILE: {}" \; -exec cat {} \; > foresight.txt`

In [11]:
import os

print(os.getcwd())

d:\research\healtheintent


In [ ]:
# find duplicate function imports in a file
import re

# Read the file
with open('lhn/__init__.py', 'r') as f:
    content = f.read()

# Find all function imports using a regular expression
function_imports = re.findall(r'import (\w+)', content)

# Find duplicates
duplicates = set([func for func in function_imports if function_imports.count(func) > 1])

print(f"Duplicate function imports: {duplicates}")

Duplicate function imports: set()


In [14]:
import os
import glob


# Create an archive of yaml configuration files.

# Create the 'configuration' directory if it doesn't exist
os.makedirs('configuration', exist_ok=True)

# Get a list of all Python files in 'lhn' directory and its subdirectories
python_files = glob.glob('configuration/*.yaml', recursive=True)

# Open the output file
with open('configuration.txt', 'w') as f:
    # For each Python file
    for file in python_files:
        print(f"{file}")
        # Write the file name to the output file
        f.write(f'FILE: {file}\n')
        
        # Write the contents of the file to the output file
        with open(file, 'r') as file_contents:
            f.write(file_contents.read())
            f.write('\n')

configuration\config-global.yaml
configuration\config_IUH.yaml
configuration\config_RWD.yaml


In [2]:
import sys
print(sys.executable)

c:\Program Files\Python310\python.exe


In [2]:

#Pack a module

import os
import glob

# Get a list of all Python files in 'lhn' directory and its subdirectories
python_files = glob.glob('lhn/**/*.py', recursive=True)
print(f"writting to {os.path.join(os.getcwd(),'lhn.txt')}")
# Open the output file
with open('lhn.txt', 'w') as f:
    # For each Python file
    for file in python_files:
        # Write the file name to the output file
        f.write(f'FILE: {file}\n')
        
        # Write the contents of the file to the output file
        with open(file, 'r') as file_contents:
            f.write(file_contents.read())
            f.write('\n')

writting to /Users/harlananelson/Documents/IUHealth/healtheintent/lhn.txt


In [63]:
print(f"writting to {os.path.join(os.getcwd(),'lhn.txt')}")

writting to d:\research\healtheintent\lhn.txt


In [35]:
def find_functions_in_directory(directory_path):
    for root, dirs, files in os.walk(directory_path):
        for file in files:
            if file.endswith('.py'):
                file_path = os.path.join(root, file)
                functions = find_functions_in_file(file_path)
                if functions:
                    module_path = os.path.relpath(file_path, directory_path)
                    module_path = os.path.splitext(module_path)[0].replace(os.sep, '.')
                    print(f'from .{module_path} import {", ".join(functions)}')
                    
find_functions_in_directory('lhn')

from .archive import strip_outputs_from_ipynb, concatenate_files, unpack_files, run_concat, run_upack, archive_subdirectories, combine_all_archives, unpack_all_archives
from .cohort import group_ethnicities, group_races, group_races2, group_gender, group_marital_status, assign_age_group, assign_age_group_, calcUsage, write_index_table, identifyLevel, select_elements_for_encounter_id, entity2Elements, createSchemaFromPd, select_by_entity, elementList2TBL, applyTimeBoundary, findConditionsByEncounter, identify_target_records
from .cohort_matching import call_match_controls, match_controls_to_cases
from .database_operations import search_object, get_id_columns, create_id_columns_dict, use_database_my, set_database, set_database, drop_table, load_table, recycleTBL, rename_tables, csv2DF, unionAll
from .data_conversion import list2YAML
from .data_display import print_parameters, print_pd, showIU, noRowNum, show_first
from .data_summary import count_people, attrition, groupCount, countDistin

## Unpack 

In [ ]:
import os
from pathlib import Path

# Open the lhn.txt file
with open('lhn.txt', 'r') as f:
    content = f.read()

# Split the content by the file headers
files = content.split('FILE: ')[1:]

for file in files:
    # Split the file content from the file path
    file_path, file_content = file.split('\n', 1)
    print(f"the file is {file_path}")

    # Remove the 'lhn\' part from the file path
    file_path = file_path.replace('lhn\\', '')

    # Create a Path object
    file_path = Path('.') / 'lhn' / file_path

    # Convert the Path object to a string
    file_path_str = str(file_path)

    # Create the directories if they don't exist
    os.makedirs(os.path.dirname(file_path_str), exist_ok=True)

    # Write the file content to the file
    with open(file_path_str, 'w') as f:
        f.write(file_content)

### To unpack

In [103]:
import os

# Open the bunsen.txt file
with open('bunsen.txt', 'r') as f:
    content = f.read()

# Split the content by the file headers
files = content.split('# FILE: ')[1:]

for file in files:
    # Split the file content from the file path
    file_path, file_content = file.split('\n', 1)

    # Remove the /usr/local/spark/python/bunsen/ part of the file path
    file_path = file_path.replace('/usr/local/spark/python/bunsen/', '')

    # Prepend './bunsen/' to the file path
    file_path = r'./bunsen/' + file_path

    # Create the directories if they don't exist
    os.makedirs(os.path.dirname(file_path), exist_ok=True)

    # Write the file content to the file
    with open(file_path, 'w') as f:
        f.write(file_content)